# Retail Sales and Customer Analysis

An end-to-end applied data project using the UCI Online Retail dataset. We inspect raw invoice lines, document data cleaning, derive business metrics, visualize results, and evaluate a chronological daily-sales forecast.

**Questions**
- Which months, countries, and products contribute most to sales?
- What proportion of identified customers return?
- Do lagged sales features beat a weekly seasonal-naive baseline on unseen later dates?

Source: [UCI Online Retail](https://doi.org/10.24432/C5BW33), Daqing Chen (2015), CC BY 4.0. The dataset covers 1 Dec 2010–9 Dec 2011. See `README.md` for methodology and caveats.

## 1. Load and inspect

The workbook is included beside this notebook. A cancelled invoice is marked by an invoice number beginning with `C`. Customer IDs are missing on some lines.

In [ ]:
import pandas as pd
from pathlib import Path
data_path = Path('Online_Retail.xlsx')
raw = pd.read_excel(data_path)
print(raw.shape)
raw.head()

## 2. Clean and engineer transaction measures

The core sales analysis excludes cancellations, nonpositive quantities/prices, and unknown customer IDs. This makes the customer metrics interpretable, but the sales total is not net of returns. The original workbook is preserved. The full, documented analysis and model are in `analysis.py`.

In [ ]:
raw['InvoiceDate'] = pd.to_datetime(raw['InvoiceDate'])
raw['Revenue'] = raw['Quantity'] * raw['UnitPrice']
is_cancelled = raw['InvoiceNo'].astype(str).str.startswith('C')
sales = raw.loc[~is_cancelled & raw['CustomerID'].notna() & (raw['Quantity'] > 0) & (raw['UnitPrice'] > 0)].copy()
sales['Revenue'] = sales['Quantity'] * sales['UnitPrice']
print(f'Usable lines: {len(sales):,}; customers: {sales.CustomerID.nunique():,}; orders: {sales.InvoiceNo.nunique():,}')

## 3. Run analysis, forecast, and charts

The script computes sales and customer summaries, plots monthly/daily sales, ranks countries/products, and evaluates a random forest on the final 20% of days against a seven-day naive baseline. All feature values are based on the past. Metrics are printed and saved to `results.json`; plots are saved in `figures/`.

Run this notebook from its containing folder.

In [ ]:
%run analysis.py

## 4. Findings and interpretation

The observed sales are concentrated in the UK, with a late-year peak. A majority of identified customers purchase more than once. The machine-learning model improves on the weekly-naive baseline on this particular holdout, but the dataset spans just over one year; the result is not a reliable promise of future accuracy. Product and customer segments are descriptive, not causal or predictive. See `README.md` for the result summary and full limitations.